# Rigvid

**Repo:** https://github.com/shivanshpatel35/rigvid  ·  **Paper:** *Robotic Manipulation by Imitating Generated Videos Without Physical Demonstrations* (Patel et al.)



## 1. Tóm tắt paper

### 1.1 Paper 2 – RIGVid 
**Bài toán.** Dạy robot thao tác vật thể (đổ, nâng, lau, quét…) *không cần demo vật lý* của người hay robot, chỉ từ ảnh RGB-D ban đầu và câu lệnh ngôn ngữ.

**Ý tưởng chính.** Dùng video diffusion model (KlingAI) sinh video "robot/người làm tác vụ", rồi *trích quỹ đạo 6-DoF của vật* từ video sinh ra và bắt robot bắt chước quỹ đạo đó:
1. Sinh video từ ảnh RGB-D đầu + prompt (và lọc video xấu bằng VLM).
2. **Depth đơn mắt từng frame** bằng RollingDepth (video depth, nhất quán theo thời gian).
3. **Pose 6-DoF**: FoundationPose đăng ký (register) ở frame đầu bằng depth thật, sau đó *track* qua các frame.
4. Làm mượt quỹ đạo, đổi sang world frame, retarget sang end-effector: `T_world_to_eef = T_world_to_object_smooth @ T_object_to_eef`.

Repo public chỉ chạy phần 2-3 trong **camera frame**.



## 2. Setup codebase


In [1]:
!nvidia-smi | head -15
import sys, torch
print(sys.version); print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.is_available())

Sat Oct 17 05:27:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content
!rm -rf rigvid
!git clone -q https://github.com/shivanshpatel35/rigvid.git
%cd /content/rigvid
!git clone -q https://github.com/prs-eth/RollingDepth.git
!git clone -q https://github.com/NVlabs/FoundationPose.git
!ls -la media media/mesh

/content
/content/rigvid
media:
total 2544
drwxr-xr-x 3 root root    4096 Oct 17 05:27 .
drwxr-xr-x 7 root root    4096 Oct 17 05:27 ..
-rw-r--r-- 1 root root      89 Oct 17 05:27 cam_K.txt
-rw-r--r-- 1 root root 1843200 Oct 17 05:27 depth.raw
-rw-r--r-- 1 root root    4741 Oct 17 05:27 mask.png
drwxr-xr-x 2 root root    4096 Oct 17 05:27 mesh
-rw-r--r-- 1 root root  735855 Oct 17 05:27 start_img.png

media/mesh:
total 10428
drwxr-xr-x 2 root root     4096 Oct 17 05:27 .
drwxr-xr-x 3 root root     4096 Oct 17 05:27 ..
-rw-r--r-- 1 root root      194 Oct 17 05:27 material.mtl
-rw-r--r-- 1 root root   451759 Oct 17 05:27 material.png
-rw-r--r-- 1 root root 10207844 Oct 17 05:27 mesh.obj


### 2A. Cài RollingDepth


In [3]:
# import torch
# tv = __import__('torchvision').__version__
# open('/content/rigvid/constraints.txt','w').write(
#     f"torch=={torch.__version__}\ntorchvision=={tv}\nnumpy==1.26.4\nscipy==1.12.0\n")
# !pip -q install numpy==1.26.4 scipy==1.12.0 PyJWT pybind11 gdown plotly
# !cd /content/rigvid/RollingDepth && pip -q install -r requirements.txt -c ../constraints.txt
# !cd /content/rigvid/RollingDepth && bash script/install_diffusers_dev.sh

In [6]:
!pip -q install -e /content/rigvid/RollingDepth/diffusers
!pip -q install omegaconf einops tqdm imageio imageio-ffmpeg opencv-python matplotlib accelerate transformers scipy plotly gdown pybind11

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for diffusers (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.4/319.4 kB 12.4 MB/s eta 0:00:00


In [4]:
# ===== ENV: chạy lại ô này sau mỗi lần restart =====
import os, sys, glob, json, time
os.chdir('/content/rigvid')
for p in ['/content/rigvid', '/content/rigvid/RollingDepth', '/content/rigvid/FoundationPose']:
    if p not in sys.path: sys.path.insert(0, p)
os.environ['PYTHONPATH'] = ':'.join(sys.path)
os.makedirs('outputs', exist_ok=True)
RESULTS = json.load(open('outputs/results.json')) if os.path.exists('outputs/results.json') else {}
def save_results(): json.dump(RESULTS, open('outputs/results.json','w'), indent=2)
import numpy as np, cv2, torch
print(np.__version__, torch.cuda.is_available())

2.1.3 True


### 2B. Cài FoundationPose


In [5]:
# %%bash
# set -x
# cd /content/rigvid/FoundationPose
# apt-get -qq install -y libeigen3-dev cmake > /dev/null
# sed -i '/^torch==.*+cu118/d; /^torchvision==.*+cu118/d; /^torchaudio==.*+cu118/d' requirements.txt
# pip -q install -r requirements.txt -c ../constraints.txt || echo "REQ_FAIL"
# pip -q install --no-cache-dir git+https://github.com/NVlabs/nvdiffrast.git fvcore iopath ninja || echo "NVDIFF_FAIL"
# export FORCE_CUDA=1 TORCH_CUDA_ARCH_LIST="7.5;8.0;8.6;8.9"
# pip -q install git+https://github.com/facebookresearch/pytorch3d.git || echo "P3D_FAIL"

Process is terminated.


In [7]:
# Patch setup.py giống install.sh rồi build CUDA extensions
setup_py = '''from setuptools import setup
import os
from torch.utils.cpp_extension import BuildExtension, CUDAExtension
code_dir = os.path.dirname(os.path.realpath(__file__))
cxx_flags = ['-O3', '-std=c++17']
nvcc_flags = ['-O3','--expt-relaxed-constexpr','-std=c++17','-U__CUDA_NO_HALF_OPERATORS__','-U__CUDA_NO_HALF_CONVERSIONS__','-U__CUDA_NO_HALF2_OPERATORS__']
setup(name='common', ext_modules=[
  CUDAExtension('common', ['bindings.cpp','common.cu'], extra_compile_args={'cxx': cxx_flags,'nvcc': nvcc_flags}),
  CUDAExtension('gridencoder', [f"{code_dir}/torch_ngp_grid_encoder/gridencoder.cu", f"{code_dir}/torch_ngp_grid_encoder/bindings.cpp"], extra_compile_args={'cxx': cxx_flags,'nvcc': nvcc_flags}),
 ], include_dirs=["/usr/local/include/eigen3","/usr/include/eigen3"], cmdclass={'build_ext': BuildExtension})
'''
open('/content/rigvid/FoundationPose/bundlesdf/mycuda/setup.py','w').write(setup_py)
import pybind11
os.environ['CMAKE_PREFIX_PATH'] = pybind11.get_cmake_dir() + ':' + os.environ.get('CMAKE_PREFIX_PATH','')
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5;8.0;8.6;8.9'
!cd /content/rigvid/FoundationPose && bash build_all_conda.sh 2>&1 | tail -15

build_all_conda.sh: line 8: CONDA_PREFIX: Set CONDA_PREFIX by activating your conda env first.


In [8]:
# Weights của FoundationPose (refiner + scorer).
!cd /content/rigvid/FoundationPose && mkdir -p weights && gdown --folder -q https://drive.google.com/drive/folders/1DFezOAD0oD1BblsXVxqDsl8fj0qzB82i -O weights
!find /content/rigvid/FoundationPose/weights -maxdepth 2 | head

/content/rigvid/FoundationPose/weights
/content/rigvid/FoundationPose/weights/2024-01-11-20-02-45
/content/rigvid/FoundationPose/weights/2024-01-11-20-02-45/config.yml
/content/rigvid/FoundationPose/weights/2024-01-11-20-02-45/model_best.pth
/content/rigvid/FoundationPose/weights/2023-10-28-18-33-37
/content/rigvid/FoundationPose/weights/2023-10-28-18-33-37/config.yml
/content/rigvid/FoundationPose/weights/2023-10-28-18-33-37/model_best.pth


> Không cài môi trường được. Nên chưa reproduce paper này được.

## 3. Training / fine-tuning
RIGVid **không có bước training**: toàn bộ model (Kling, RollingDepth, FoundationPose) là pretrained, repo chỉ cung cấp inference. Vì vậy phần "small-scale" được thay bằng **ablation inference** nhỏ trên cùng dữ liệu demo:
- RollingDepth preset `fast` (mặc định) so với `full` (nếu bật `RUN_FULL`, tốn thời gian/VRAM hơn),
- ảnh hưởng của **smoothing** quỹ đạo (§5.2), đúng điều README nhấn mạnh là "crucial".

## 4. Depth từ video sinh ra (RollingDepth)
### 4.1 Rescale video và inference

In [9]:
# from utils import rescale_video, prepare_fp_inputs
# from depth_predictor import DepthPredictor

# vids = sorted(glob.glob('media/*.mp4')); print('videos:', vids)
# INPUT_VIDEO = vids[0]
# stem = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
# RESCALED = f'outputs/{stem}_rescaled.mp4'
# rescale_video(input_path=INPUT_VIDEO, output_path=RESCALED, width=1280, height=720)

# cap = cv2.VideoCapture(RESCALED)
# T = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS); cap.release()
# print('frames', T, 'fps', fps)

# t0 = time.time()
# predictor = DepthPredictor(checkpoint='prs-eth/rollingdepth-v1-0', device='cuda', dtype='fp16', color_maps=['Greys_r'])
# depth_npy = predictor.predict(input_video_path=RESCALED, output_dir='outputs')
# if depth_npy is None:
#     depth_npy = np.load(sorted(glob.glob('outputs/*_pred.npy'))[0])
# t_depth = time.time() - t0
# depth_npy = np.asarray(depth_npy)
# print('depth', depth_npy.shape, depth_npy.dtype, float(depth_npy.min()), float(depth_npy.max()), f'{t_depth:.1f}s')
# RESULTS.update(video=INPUT_VIDEO, n_frames=T, depth_time_s=round(t_depth,1), depth_shape=list(depth_npy.shape)); save_results()

ImportError: cannot import name 'AutoencoderKL' from 'diffusers' (unknown location)

### 4.2 Đọc GT depth (frame đầu, từ camera thật) và căn chỉnh affine

In [10]:
H, W = 720, 1280
raw = open('media/depth.raw','rb').read(); n = len(raw)
if n == H*W*4:   gt = np.frombuffer(raw, np.float32).reshape(H,W).copy()
elif n == H*W*2: gt = np.frombuffer(raw, np.uint16).reshape(H,W).astype(np.float32)/1000.0
else: raise ValueError(f'Không đoán được định dạng depth.raw (size={n}). Sửa tay ở đây.')
gt[~np.isfinite(gt)] = 0
print('GT depth range (m):', gt[gt>0].min(), gt[gt>0].max())
obj_mask = cv2.imread('media/mask.png', 0); obj_mask = cv2.resize(obj_mask, (W,H), interpolation=cv2.INTER_NEAREST) > 0

def get_frame_depth(i):
    d = depth_npy[i]
    if d.ndim == 3: d = d[...,0]
    return cv2.resize(d.astype(np.float32), (W,H), interpolation=cv2.INTER_LINEAR)

def affine_align(pred, gt, valid):
    A = np.stack([pred[valid], np.ones(valid.sum())], 1)
    (a, b), *_ = np.linalg.lstsq(A, gt[valid], rcond=None)
    return a*pred + b

def depth_metrics(aligned, gt, valid):
    p, g = np.clip(aligned[valid], 1e-3, None), gt[valid]
    absrel = np.mean(np.abs(p-g)/g); rmse = np.sqrt(np.mean((p-g)**2))
    d1 = np.mean(np.maximum(p/g, g/p) < 1.25)
    return dict(AbsRel=float(absrel), RMSE_m=float(rmse), delta1=float(d1), corr=float(np.corrcoef(p,g)[0,1]))

GT depth range (m): 0.332 1.73


### 4.3 Metric định lượng: AbsRel / RMSE / δ<1.25 (frame 0 vs GT RGB-D)

In [11]:
# valid = (gt > 0.1) & (gt < 5.0)
# pred0 = get_frame_depth(0)
# al0 = affine_align(pred0, gt, valid)
# m_all = depth_metrics(al0, gt, valid)
# m_obj = depth_metrics(al0, gt, valid & obj_mask)
# print('Toàn ảnh :', m_all); print('Vùng vật :', m_obj)

# # Temporal flicker: thay đổi frame-to-frame trên nền (ngoài mask) sau khi căn affine theo frame 0
# bg = valid & ~cv2.dilate(obj_mask.astype(np.uint8), np.ones((51,51),np.uint8)).astype(bool)
# a_ref = np.polyfit(pred0[valid], gt[valid], 1)
# idx = np.linspace(0, len(depth_npy)-1, min(len(depth_npy), 30)).astype(int)
# seq = [np.polyval(a_ref, get_frame_depth(i))[bg] for i in idx]
# flicker = float(np.mean([np.mean(np.abs(seq[k]-seq[k-1])) for k in range(1,len(seq))]))
# print(f'Temporal flicker nền (m, giữa các frame lấy mẫu): {flicker:.4f}')
# RESULTS.update(depth_frame0_all=m_all, depth_frame0_object=m_obj, bg_flicker_m=flicker); save_results()

NameError: name 'depth_npy' is not defined

### 4.4 Visualize depth

In [12]:
import matplotlib.pyplot as plt
cap = cv2.VideoCapture(RESCALED)
def read_frame(i):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, f = cap.read()
    return cv2.cvtColor(f, cv2.COLOR_BGR2RGB) if ok else None

vmax = np.percentile(gt[valid], 98)
err = np.where(valid, np.abs(al0-gt), np.nan)
fig, ax = plt.subplots(2, 3, figsize=(17,8))
ax[0,0].imshow(read_frame(0)); ax[0,0].set_title('Generated video, frame 0')
ax[0,1].imshow(np.where(valid,gt,np.nan), vmin=0, vmax=vmax, cmap='turbo'); ax[0,1].set_title('GT depth (m)')
ax[0,2].imshow(np.clip(al0,0,vmax), vmin=0, vmax=vmax, cmap='turbo'); ax[0,2].set_title('RollingDepth (affine-aligned)')
im = ax[1,0].imshow(err, cmap='magma', vmin=0, vmax=0.2); ax[1,0].set_title('|error| (m)'); plt.colorbar(im, ax=ax[1,0], fraction=0.046)
for k, fi in enumerate([len(depth_npy)//2, len(depth_npy)-1]):
    ax[1,k+1].imshow(np.clip(np.polyval(a_ref, get_frame_depth(fi)),0,vmax), cmap='turbo', vmin=0, vmax=vmax); ax[1,k+1].set_title(f'Predicted depth, frame {fi}')
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.savefig('outputs/fig_depth.png', dpi=110); plt.show()

NameError: name 'RESCALED' is not defined

In [13]:
# (Tuỳ chọn) Ablation preset 'full'. Mặc định tắt vì lâu / dễ OOM trên T4.
RUN_FULL = False
if RUN_FULL:
    import subprocess
    t0 = time.time()
    subprocess.run(['python','RollingDepth/run_video.py','-i',RESCALED,'-o','outputs/full','-p','full','--verbose'], check=False)
    t_full = time.time()-t0
    f = glob.glob('outputs/full/**/*_pred.npy', recursive=True)
    if f:
        dn = np.load(f[0]); d = cv2.resize(dn[0].astype(np.float32), (W,H))
        mf = depth_metrics(affine_align(d, gt, valid), gt, valid)
        print('full:', mf, f'{t_full:.0f}s vs fast {t_depth:.0f}s'); RESULTS.update(depth_full=mf, depth_full_time_s=round(t_full,1)); save_results()

## 5. Pose 6-DoF (FoundationPose) và quỹ đạo
### 5.1 Chuẩn bị input và chạy rollout

In [14]:
prepare_fp_inputs(
    rescaled_video_path=RESCALED, mask_path='media/mask.png', cam_intrinsics_path='media/cam_K.txt',
    gt_depth_path='media/depth.raw', depth_npy=depth_npy, output_dir='outputs',
    window_size=20, width=1280, height=720)
print(sorted(os.listdir('outputs')))

NameError: name 'RESCALED' is not defined

In [15]:
POSE_OK = False
try:
    from pose_rollout_predictor import PoseRolloutPredictor
    t0 = time.time()
    PoseRolloutPredictor(data_path='outputs', mesh_file='media/mesh/mesh.obj',
                         est_refine_iter=10, track_refine_iter=4, debug=0,
                         debug_dir='outputs/fp_outputs').run()
    t_pose = time.time()-t0
    POSE_OK = len(glob.glob('outputs/fp_outputs/ob_in_cam/*.txt')) > 0
    print('pose time', round(t_pose,1), 's  OK =', POSE_OK)
    RESULTS.update(pose_time_s=round(t_pose,1))
except Exception as e:
    import traceback; traceback.print_exc()
    RESULTS['pose_error'] = repr(e)[:300]
    print('\n FoundationPose không chạy được trong môi trường này → ghi vào §6 như một failure của setup.')
RESULTS['pose_ok'] = POSE_OK; save_results()


 FoundationPose không chạy được trong môi trường này → ghi vào §6 như một failure của setup.


Traceback (most recent call last):
  File "/tmp/ipykernel_13729/2124178809.py", line 3, in <cell line: 0>
    from pose_rollout_predictor import PoseRolloutPredictor
  File "/content/rigvid/pose_rollout_predictor.py", line 15, in <module>
    import trimesh
ModuleNotFoundError: No module named 'trimesh'
